# MultiLexNorm 2026 Codabench Playground

Use this notebook to load the MultiLexNorm development dataset, experiment with predictions, and export a Codabench-ready submission ZIP.

The exported file matches the sample repository format: a flat ZIP containing `predictions.json`, where each JSON record has `raw`, `norm`, `lang`, and `pred` fields.

## 1. Imports and output paths

In [ ]:
from pathlib import Path
import json
import sys
import zipfile

import pandas as pd
from datasets import concatenate_datasets, load_dataset, load_from_disk

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils.py").exists():
    matches = list(PROJECT_ROOT.rglob("utils.py"))
    if matches:
        PROJECT_ROOT = matches[0].parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from utils import counting, evaluate, mfr
except ModuleNotFoundError:
    def counting(data):
        counts = {}
        for item in data:
            for word_raw, word_gold in zip(item["raw"], item["norm"]):
                counts.setdefault(word_raw, {})
                counts[word_raw][word_gold] = counts[word_raw].get(word_gold, 0) + 1
        return counts

    def mfr(input_sent, counts):
        counts = counts or {}
        return [max(counts[word], key=counts[word].get) if word in counts else word for word in input_sent]

    def evaluate(raw, gold, pred, ignCaps=False, verbose=False, info=True):
        cor = changed = total = 0
        if len(gold) != len(pred):
            raise ValueError("Gold normalization and prediction contain different numbers of sentences")
        for sent_raw, sent_gold, sent_pred in zip(raw, gold, pred):
            if len(sent_gold) != len(sent_pred):
                raise ValueError("A sentence has different gold/pred lengths")
            for word_raw, word_gold, word_pred in zip(sent_raw, sent_gold, sent_pred):
                if ignCaps:
                    word_raw, word_gold, word_pred = word_raw.lower(), word_gold.lower(), word_pred.lower()
                changed += word_raw != word_gold
                cor += word_gold == word_pred
                if word_gold != word_pred and verbose:
                    print(word_raw, word_gold, word_pred)
                total += 1
        accuracy = cor / total
        lai = (total - changed) / total
        err = (accuracy - lai) / (1 - lai)
        if info:
            print("Baseline acc.(LAI): {:.2f}".format(lai * 100))
            print("Accuracy:           {:.2f}".format(accuracy * 100))
            print("ERR:                {:.2f}".format(err * 100))
        return lai, accuracy, err

print(f"Project root: {PROJECT_ROOT}")

DATASET_ID = "weerayut/multilexnorm2026-dev-pub"
LOCAL_DATA_DIR = Path("data/multilexnorm2026-dev-pub")

SAVE_DIR = Path("outputs/playground_submission")
PREDICTIONS_PATH = SAVE_DIR / "predictions.json"
ZIP_PATH = SAVE_DIR.with_suffix(".zip")

SAVE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the dataset

The notebook first tries the local copy downloaded into `data/multilexnorm2026-dev-pub`. If it is not present, it downloads the dataset from Hugging Face and saves it locally.

In [ ]:
def load_multilexnorm(dataset_id=DATASET_ID, local_dir=LOCAL_DATA_DIR):
    if local_dir.exists():
        print(f"Loading dataset from disk: {local_dir}")
        return load_from_disk(str(local_dir))

    print(f"Downloading dataset from Hugging Face: {dataset_id}")
    data = load_dataset(dataset_id)
    local_dir.parent.mkdir(parents=True, exist_ok=True)
    data.save_to_disk(str(local_dir))
    return data


data = load_multilexnorm()
data

In [ ]:
for split_name, split_data in data.items():
    langs = sorted(set(split_data["lang"]))
    print(f"{split_name:10s} rows={len(split_data):6d} languages={len(langs):2d} {langs}")

pd.DataFrame(data["train"][:5])

## 3. Evaluate the simple MFR baseline

Change `LANG` to inspect a different language. This section trains a Most Frequent Replacement dictionary on the selected language's train split and evaluates on that language's validation split.

In [ ]:
LANG = "en"

train_df_all = data["train"].to_pandas()
val_df_all = data["validation"].to_pandas()

train_lang_df = train_df_all.loc[train_df_all["lang"] == LANG].copy()
val_df = val_df_all.loc[val_df_all["lang"] == LANG].copy()

if train_lang_df.empty or val_df.empty:
    raise ValueError(f"No train/validation data found for LANG={LANG!r}")

counts = counting(train_lang_df.to_dict(orient="records"))
sample = ["bcause", "u", "r", "funny"]
print("Smoke test:", sample, "->", mfr(sample, counts))

val_df["pred"] = val_df["raw"].apply(lambda sent: mfr(sent, counts))
_ = evaluate(val_df["raw"].tolist(), val_df["norm"].tolist(), val_df["pred"].tolist())

val_df[["raw", "norm", "pred", "lang"]].head()

## 4. Build predictions for Codabench

For the public dev submission, train on `train + validation` and predict the `test` split. The helper below builds a separate MFR dictionary per language, then predicts each test row with the dictionary for that row's language.

In [ ]:
def predict_mfr_by_language(train_split, target_split):
    train_df = train_split.to_pandas()
    target_df = target_split.to_pandas()

    counts_by_lang = {}
    for lang, train_lang_df in train_df.groupby("lang"):
        counts_by_lang[lang] = counting(train_lang_df.to_dict(orient="records"))

    def predict_row(row):
        counts = counts_by_lang.get(row["lang"], {})
        return mfr(row["raw"], counts)

    target_df["pred"] = target_df.apply(predict_row, axis=1)
    return target_df


def as_token_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if hasattr(value, "tolist"):
        return value.tolist()
    raise TypeError(f"Expected a token sequence, found {type(value)}")


def ensure_norm_column(df):
    df = df.copy()
    df["raw"] = df["raw"].apply(as_token_list)
    if "pred" in df.columns:
        df["pred"] = df["pred"].apply(as_token_list)

    if "norm" not in df.columns:
        df["norm"] = df["raw"].apply(lambda sent: [""] * len(sent))
    else:
        df["norm"] = df.apply(
            lambda row: as_token_list(row["norm"])
            if len(as_token_list(row["norm"])) == len(row["raw"])
            else [""] * len(row["raw"]),
            axis=1,
        )
    return df


def validate_submission(df, reference_split):
    required_columns = ["raw", "norm", "lang", "pred"]
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if len(df) != len(reference_split):
        raise ValueError(f"Expected {len(reference_split)} rows, found {len(df)}")

    for row_idx, row in df[required_columns].iterrows():
        raw = as_token_list(row["raw"])
        norm = as_token_list(row["norm"])
        pred = as_token_list(row["pred"])
        if not isinstance(raw, list) or not isinstance(norm, list) or not isinstance(pred, list):
            raise TypeError(f"Row {row_idx}: raw, norm, and pred must all be token lists")
        if len(raw) != len(norm) or len(raw) != len(pred):
            raise ValueError(
                f"Row {row_idx}: length mismatch raw={len(raw)}, norm={len(norm)}, pred={len(pred)}"
            )
        if not isinstance(row["lang"], str):
            raise TypeError(f"Row {row_idx}: lang must be a string")

    print("Submission schema looks valid.")
    return True

In [ ]:
def write_submission(df, save_dir=SAVE_DIR, predictions_path=PREDICTIONS_PATH, zip_path=ZIP_PATH):
    save_dir.mkdir(parents=True, exist_ok=True)

    output_df = df[["raw", "norm", "lang", "pred"]].copy()
    records = output_df.to_dict(orient="records")

    with predictions_path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(predictions_path, arcname="predictions.json")

    print(f"Wrote {predictions_path}")
    print(f"Wrote {zip_path}")
    return predictions_path, zip_path


submission_train = concatenate_datasets([data["train"], data["validation"]])
target_split = data["test"]

submission_df = predict_mfr_by_language(submission_train, target_split)
submission_df = ensure_norm_column(submission_df)
validate_submission(submission_df, target_split)
predictions_path, zip_path = write_submission(submission_df)

submission_df[["raw", "norm", "lang", "pred"]].head()

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    print("ZIP contents:", zf.namelist())
    with zf.open("predictions.json") as f:
        preview = json.loads(f.read().decode("utf-8"))[:2]

preview

## 5. Swap in your own model

Replace the body of `predict_with_custom_model` with your model inference. Keep the output contract the same: one `pred` token list for each input `raw` token list.

In [ ]:
def predict_with_custom_model(train_split, target_split):
    # TODO: Replace this baseline with your own model.
    return predict_mfr_by_language(train_split, target_split)


# custom_df = predict_with_custom_model(submission_train, target_split)
# custom_df = ensure_norm_column(custom_df)
# validate_submission(custom_df, target_split)
# write_submission(custom_df)